In [1]:
import ee
import geemap

In [2]:
PROJECT_ID = "solar-grail-510708-p8"

ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized successfully.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized successfully.


In [3]:
message = ee.String("Connected to Google Earth Engine")

print(message.getInfo())

Connected to Google Earth Engine


In [4]:
import geopandas as gpd

bengaluru_gdf = gpd.read_file("C:/Users/Ā/Documents/python_class/dataset/Bengaluru_boundary.geojson")

bengaluru_gdf

,objectid,statename,statecode,districtname,districtcode,ulbname,ulbcode,wardname,wardcode,id,remark,st_area(shape),st_perimeter(shape),geometry
0,104610,Karnataka,29,Bengaluru Urban,572,Bruhat Bengaluru Mahanagara Palike,803162,Sanjaya Nagar,19,None,Approved,0.000128,0.059817,"POLYGON ((77.54827 12.86691, 77.54733 12.86664..."


In [5]:
bengaluru = geemap.geopandas_to_ee(
    bengaluru_gdf
)

bengaluru

In [6]:
landsat8 = ee.ImageCollection(
    "LANDSAT/LC08/C02/T1_L2"
)

landsat9 = ee.ImageCollection(
    "LANDSAT/LC09/C02/T1_L2"
)

landsat = landsat8.merge(
    landsat9
)

landsat_hot_season = (
    landsat
    .filterBounds(bengaluru)
    .filterDate(
        "2025-03-01",
        "2025-06-01"
    )
    .filter(
        ee.Filter.lt(
            "CLOUD_COVER",
            50
        )
    )
    .filter(
        ee.Filter.eq(
            "PROCESSING_LEVEL",
            "L2SP"
        )
    )
)

print(
    "Landsat scenes:",
    landsat_hot_season.size().getInfo()
)

Landsat scenes: 9


In [7]:
def prepare_landsat_lst(image):

    qa = image.select("QA_PIXEL")

    clear = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(
            qa.bitwiseAnd(1 << 2).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 3).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 4).eq(0)
        )
        .And(
            qa.bitwiseAnd(1 << 5).eq(0)
        )
    )

    lst_c = (
        image
        .select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("LST_C")
    )

    return (
        lst_c
        .updateMask(clear)
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )

#The lecture uses ST_B10, applies the scale factor, adds 149, then converts Kelvin to °C by subtracting 273.15.

In [8]:
landsat_lst = landsat_hot_season.map(
    prepare_landsat_lst
)

lst = (
    landsat_lst
    .median()
    .clip(bengaluru)
)

lst

In [9]:
lst_vis = {
    "min": 20,
    "max": 45,
    "palette": [
        "navy",
        "blue",
        "cyan",
        "yellow",
        "orange",
        "red",
        "darkred"
    ]
}

Map = geemap.Map()

Map.centerObject(bengaluru, 10)

Map.addLayer(
    lst,
    lst_vis,
    "Median LST: Mar–May 2025"
)

Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Study area"
)

Map

Map(center=[12.977822480314208, 77.60125519188976], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
lst_stats = lst.reduceRegion(
    reducer=(
        ee.Reducer.min()
        .combine(
            ee.Reducer.mean(),
            sharedInputs=True
        )
        .combine(
            ee.Reducer.max(),
            sharedInputs=True
        )
    ),
    geometry=bengaluru,
    scale=30,
    maxPixels=1e9
)

lst_stats.getInfo()

{'LST_C_max': 51.351654920000016,
 'LST_C_mean': 38.369185836008825,
 'LST_C_min': 27.992906240000025}

In [11]:
# Define the high-heat threshold
heat_threshold = 40

# Identify pixels where LST is greater than 35°C
high_heat = (
    lst
    .gt(heat_threshold) ## greater than
    .selfMask()
    .rename("High_Heat")
)

In [12]:

Map = geemap.Map()

Map.centerObject(
    bengaluru,
    10
)

# Add the complete LST surface
Map.addLayer(
    lst,
    lst_vis,
    "Land Surface Temperature"
)

# Add only pixels above 40°C
Map.addLayer(
    high_heat,
    {"palette": ["red"]},
    "LST > 40°C"
)

# Add Bengaluru boundary
Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Bengaluru Boundary"
)

Map

Map(center=[12.977822480314032, 77.60125519188985], controls=(WidgetControl(options=['position', 'transparent_…

In [13]:
population = (
    ee.ImageCollection(
        "WorldPop/GP/100m/pop"
    )
    .filterBounds(bengaluru)
    .filterDate(
        "2020-01-01",
        "2021-01-01"
    )
    .mosaic()
    .select("population")
    .clip(bengaluru)
)

population

In [15]:
import geemap.colormaps as cm

population_vis = {
    "min": 0,
    "max": 500,
    "palette": cm.get_palette(
        "YlOrRd",
        n_class=6
    )
}

Map = geemap.Map()

Map.centerObject(
    bengaluru,
    10
)

Map.addLayer(
    population,
    population_vis,
    "WorldPop 2020"
)

Map.addLayer(
    high_heat,
    {"palette": ["cyan"]},
    "LST > 40°C"
)

Map.addLayer(
    bengaluru,
    {"color": "white"},
    "Bengaluru Boundary"
)

Map

Map(center=[12.977822480314032, 77.60125519188985], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
utci and world population data then 